In [ ]:
"""
Task is to turn redundancy_stimuli.json into probe prompts for the 3-step diagnostic. for all 4 conditions (2x2  prior x utterance type), in both chat-format and few-shot-completion-format.

Each probe  also has "mcq" variant: the same question reformatted as lettered multiple choice (A/B), with which meaning maps to which letter RANDOMIZED per item+step (seeded, so reproducible).
This avoids surface form competition (derived from Holtzman et al. 2021), which is bacsically scoring a single-token letter continuation instead of a multi-word phrase, since phrases can have unequal baseline probability
 independent of content, which multi-token log-prob comparison conflates with a real answer.
"""
# LLM were used to write the sentences, half given and later checked by me. and the later part of the code was given to the LLM for better and presentable formatting. the original code canbe presented on ask.
import json
import random

# target_property per item
# written a  short, positive, generic phrasing of the habitual/critical detail matched to what each item's wonky_context negates.(written manually with help from LLMs to avoid errors)
TARGET_PROPERTY = {
    1:  "he pays the cashier when he goes to the grocery store",
    2:  "she eats at the restaurants she visits",
    3:  "he keeps the common areas of the apartment clean",
    4:  "she pays for her subway rides",
    5:  "he pays attention to what he's doing when he fuels up",
    6:  "she remembers to bring the things she needs when swimming",
    7:  "he gets off at his stop on the train",
    8:  "she remembers her things when leaving the house",
    9:  "he gets examined by the doctor at the medical practice",
    10: "she uses shampoo to wash her hair",
    11: "he actually jumps out of the plane during skydiving courses",
    12: "she mails the letters she writes",
    13: "he walks to the bus stop",
    14: "she tries on clothes when shopping with friends",
    15: "he boils pasta in water",
    16: "she preheats the oven when baking",
    17: "he carries his phone on board rather than checking it in",
    18: "she uses scissors to cut her own hair",
    19: "he looks closely at the art when he goes to exhibitions",
    20: "she locks her car after parking",
    21: "he gets to pick the toppings on his pizza",
    22: "she rinses the dishes after washing them",
    23: "she checks books out of the library rather than taking them",
    24: "he uses detergent when doing the laundry",
}

WONKY_PROPERTY = {
    1:  "he is usually broke and doesn't pay when he shops",
    2:  "she doesn't actually enjoy eating at restaurants",
    3:  "he doesn't keep the common areas of the apartment clean",
    4:  "she doesn't pay for her subway rides",
    5:  "he doesn't pay attention to what he's doing when he fuels up",
    6:  "she often forgets to bring the things she needs",
    7:  "he often misses his stop on the train",
    8:  "she is scatter-brained in everyday life",
    9:  "he usually only sees the nurse, not the doctor",
    10: "she uses a cleansing conditioner only, not shampoo",
    11: "he is still terrified of heights",
    12: "she keeps her letters rather than mailing them",
    13: "he bikes to the bus stop rather than walking",
    14: "she finds clothes shopping boring and doesn't try anything on",
    15: "he prefers pasta that doesn't need to be pre-boiled",
    16: "she is terrible at following basic directions in recipes",
    17: "he checks everything into the hold rather than carrying it on",
    18: "she uses a buzzer rather than scissors to cut her hair",
    19: "he only goes to exhibitions because his girlfriend drags him",
    20: "she is incredibly scatterbrained",
    21: "he doesn't usually get to choose his pizza toppings",
    22: "she doesn't always bother to rinse the dishes",
    23: "she steals books rather than checking them out",
    24: "he can't get a handle even on the basics of laundry",
}

FEWSHOT_PREFIX =


def split_utterance(utterance: str):
    #Split announcement and critical detail into  two parts
    parts = utterance.split(". ", 1)
    announcement = parts[0].strip() + "."
    critical_detail = parts[1].strip() if len(parts) > 1 else ""
    return announcement, critical_detail


def build_conditions(item: dict):
    #expand  stimulus item into the 2x2
    hab_announce, hab_detail = split_utterance(item["habitual_utterance"])
    non_announce, non_detail = split_utterance(item["non_habitual_utterance"])

    conditions = [
        {
            "condition": "critical",  # ordinary + habitual -> redundant, should trigger inference
            "context": item["ordinary_context"],
            "announcement": hab_announce,
            "critical_detail": hab_detail,
            "full_utterance": item["habitual_utterance"],
        },
        {
            "condition": "resolved",  # wonky + habitual -> informative, resolve expectation
            "context": item["wonky_context"],
            "announcement": hab_announce,
            "critical_detail": hab_detail,
            "full_utterance": item["habitual_utterance"],
        },
        {
            "condition": "baseline_ordinary",  # ordinary + non-habitual
            "context": item["ordinary_context"],
            "announcement": non_announce,
            "critical_detail": non_detail,
            "full_utterance": item["non_habitual_utterance"],
        },
        {
            "condition": "baseline_wonky",  # wonky + non-habitual
            "context": item["wonky_context"],
            "announcement": non_announce,
            "critical_detail": non_detail,
            "full_utterance": item["non_habitual_utterance"],
        },
    ]
    return conditions


def make_mcq(seed_key: str, question_text: str, option_map: dict):
    """
    Build a lettered multiple-choice version of a probe. letter assignment is randomized per (item, step) using seed_key, so order is counterbalanced across the dataset rather than fixed
    this controls for any residual letter-position bias on top of fixing the surface-form-competition issue.
    Returns: prompt text chat-style, ending at "Answer:"  and a letter_map showing which letter means what
    """
    rng = random.Random(seed_key)  # reproducible per item+step
    labels = list(option_map.keys())
    rng.shuffle(labels)
    letters = ["A", "B"]
    letter_map = {letters[i]: labels[i] for i in range(2)}  # {"A": "new", "B": "expected"}

    options_text = "\n".join(
        f"({letters[i]}) {option_map[labels[i]]}" for i in range(2)
    )
    prompt = f"{question_text}\n{options_text}\nAnswer with the letter only.\nAnswer:"
    return prompt, letter_map


def make_probes(item: dict, cond: dict):
    prop = TARGET_PROPERTY[item["item_id"]]
    ctx = cond["context"]
    narrative = item["narrative"]
    speaker, listener = item["speaker"], item["listener"]
    utterance = cond["full_utterance"]

    seed_prefix = f"item{item['item_id']}_{cond['condition']}"

    # script knowledge MCQ
    sk_question = f"{ctx}\nQuestion: based on this, is it likely or unlikely that {prop}?"
    sk_mcq_prompt, sk_letter_map = make_mcq(
        seed_prefix + "_sk", sk_question,
        {"likely": "Likely", "unlikely": "Unlikely"},
    )

    #redundancy identification MCQ
    ri_question = (
        f"{ctx} {narrative}\n"
        f"{speaker} said to {listener}: \"{cond['announcement']} {cond['critical_detail']}\"\n"
        f"question: is the detail \"{cond['critical_detail']}\" new information, "
        f"or something already expected given the context?"
    )
    ri_mcq_prompt, ri_letter_map = make_mcq(
        seed_prefix + "_ri", ri_question,
        {"new": "New information", "expected": "Already expected"},
    )

    # inference derivation MCQ
    wonky_prop_preview = WONKY_PROPERTY[item["item_id"]]
    id_question = (
        f"{ctx} {narrative}\n"
        f"{speaker} said to {listener}: \"{utterance}\"\n"
        f"question: based on this conversation, is it likely or unlikely that {wonky_prop_preview}?"
    )
    id_mcq_prompt, id_letter_map = make_mcq(
        seed_prefix + "_id", id_question,
        {"likely": "Likely", "unlikely": "Unlikely"},
    )

    # script knowledge-> depends only on context and not utterance
    sk_chat = (
        f"{ctx}\n"
        f"question: Based on this, is it likely or unlikely that {prop}? "
        f"answer with exactly one word: Likely or Unlikely."
    )
    sk_completion = FEWSHOT_PREFIX + (
        f"context: {ctx}\n"
        f"question: Based on this, is it likely or unlikely that {prop}?\n"
        f"Answer:"
    )

    # -redundancy identification
    ri_chat = (
        f"{ctx} {narrative}\n"
        f"{speaker} said to {listener}: \"{cond['announcement']} {cond['critical_detail']}\"\n"
        f"Question: Is the detail \"{cond['critical_detail']}\" new information, "
        f"or something already expected given the context? "
        f"Answer with exactly one phrase: New information or Already expected."
    )
    ri_completion = FEWSHOT_PREFIX + (
        f"Context: {ctx} {narrative}\n"
        f"Statement: \"{cond['announcement']} {cond['critical_detail']}\"\n"
        f"Question: Is the detail new information, or already expected given the context?\n"
        f"Answer:"
    )

    # inference derivation
    wonky_prop = WONKY_PROPERTY[item["item_id"]]
    id_chat = (
        f"{ctx} {narrative}\n"
        f"{speaker} said to {listener}: \"{utterance}\"\n"
        f"Question: Based on this conversation, is it likely or unlikely that {wonky_prop}? "
        f"Answer with exactly one word: Likely or Unlikely."
    )
    id_completion = FEWSHOT_PREFIX + (
        f"Context: {ctx} {narrative}\n"
        f"Statement: \"{utterance}\"\n"
        f"Question: Based on this, is it likely or unlikely that {wonky_prop}?\n"
        f"Answer:"
    )

    return {
        "script_knowledge": {
            "chat": sk_chat, "completion": sk_completion,
            "mcq": sk_mcq_prompt, "mcq_letter_map": sk_letter_map,
        },
        "redundancy_id": {
            "chat": ri_chat, "completion": ri_completion,
            "mcq": ri_mcq_prompt, "mcq_letter_map": ri_letter_map,
        },
        "inference_derivation": {
            "chat": id_chat, "completion": id_completion,
            "mcq": id_mcq_prompt, "mcq_letter_map": id_letter_map,
        },
    }


def main():
    with open("/content/drive/MyDrive/Colab Notebooks/pragmatics_project/redundancy_stimuli.json") as f:
        items = json.load(f)

    output = []
    for item in items:
        for cond in build_conditions(item):
            row = {
                "item_id": item["item_id"],
                "character": item["character"],
                "condition": cond["condition"],
                "target_property": TARGET_PROPERTY[item["item_id"]],
                "wonky_property": WONKY_PROPERTY[item["item_id"]],
                "probes": make_probes(item, cond),
            }
            output.append(row)

    with open("/content/drive/MyDrive/Colab Notebooks/pragmatics_project/redundancy_probes.json", "w") as f:
        json.dump(output, f, indent=2)

    print(f"Generated {len(output)} rows (24 items x 4 conditions) "
          f"-> /content/drive/MyDrive/Colab Notebooks/pragmatics_project/redundancy_probes.json")


if __name__ == "__main__":
    main()

Generated 96 rows (24 items x 4 conditions) -> /content/drive/MyDrive/Colab Notebooks/pragmatics_project/redundancy_probes.json
